# AnIME 专用纠错 L3 模型训练（task-07 P2）

在 Google Drive 中准备目录 `MyDrive/AnIME_l3/`：

- `train_l3_correction.py`（本目录同名脚本）
- `data/train.jsonl`、`data/val.jsonl`（本机 `python3 scripts/data_pipeline/build_l3_sft.py` 生成于 `data/compiled/l3_sft/`）

运行时类型：冒烟测试用 **T4**；正式训练用 **A100**（或 H100）。checkpoint 写入 Drive，断线后重跑同一单元格会自动续训。

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
ROOT = '/content/drive/MyDrive/AnIME_l3'
DATA = f'{ROOT}/data'
SCRIPT = f'{ROOT}/train_l3_correction.py'
import os
assert os.path.exists(SCRIPT) and os.path.exists(f'{DATA}/train.jsonl'), '请先上传脚本与数据到 ' + ROOT

## 1. 冒烟测试（约 5–10 分钟，T4 即可）
2000 条样本、50 步训练，并完整走一遍 GGUF 导出，确认数据、损失掩码、checkpoint 与转换都正常。

In [ ]:
!python "$SCRIPT" --data-dir "$DATA" --out-dir "$ROOT/smoke" --smoke

## 2. 正式训练（A100 / H100）
训练结束自动导出 `anime-l3-360m-v1.{Q4_K_M,Q8_0}.gguf` 到 `ROOT/run1/`。可调参数：`--epochs`、`--lr`、`--max-len`。

In [ ]:
!python "$SCRIPT" --data-dir "$DATA" --out-dir "$ROOT/run1" --epochs 1 --lr 1e-4 --export

## 3. 只导出（训练已完成但导出中断时）

In [ ]:
!python "$SCRIPT" --data-dir "$DATA" --out-dir "$ROOT/run1" --export-only

## 4. 取回结果
从 Drive 下载 `run1/anime-l3-360m-v1.Q4_K_M.gguf`（以及 `train_report.json`、`export_info.json`），放到本机
`data/compiled/models/`，交给 Agent 做接入与全量评测。